In [40]:
import os
from common.utils import sorted_alphanum
import torch
from torch.utils.data import Dataset, random_split, DataLoader
from PIL import Image
from tqdm import tqdm
import pandas as pd

os.getcwd()
# os.chdir(r'E:\a\workspacePyCharm\Mine_Study\Atguigu_DS\image_search')

'E:\\a\\workspacePyCharm\\Mine_Study\\Atguigu_DS\\image_search'

In [ ]:
# 定义一个字典，保存分类号和中文名的映射关系
classification_names = {
    0: '上身衣服',
    1: '鞋',
    2: '包',
    3: '下身衣服',
    4: '手表',
}

## 数据预处理

In [25]:
# 自定义数据集类
class ImageLabelDataset(Dataset):
    # 初始化：传入图片根目录，以及预处理转换操作
    def __init__(self, main_dir, transform=None):
        self.main_dir = main_dir
        self.transform = transform
        self.img_names = sorted_alphanum(os.listdir(main_dir))
        labels = pd.read_csv("./common/fashion-labels.csv")
        self.label_dict = dict( zip(labels['id'], labels['target']) )

    # 获取数据集大小
    def __len__(self):
        return len(self.img_names)

    # 根据索引号得到（input, target）
    def __getitem__(self, idx):
        # 1. 根据索引号找到文件名，读取图片数据
        img_path = os.path.join(self.main_dir, self.img_names[idx])
        img = Image.open(img_path).convert('RGB')
        # 2. 将原始图片转换为符合模型输入要求的张量,这就是重构图像的目标
        if self.transform is not None:
            img_tensor = self.transform(img)
        else:
            raise ValueError("Transform must be provided!")
        # 3. 找到图片对应的分类标签
        label = self.label_dict[idx]
        # 将输入和目标返回
        return img_tensor, label

In [26]:
import torchvision.transforms as T
# 定义数据预处理转换
transform = T.Compose([
    T.Resize((64, 64,)),
    T.ToTensor()
])
dataset = ImageLabelDataset("./common/dataset", transform=transform)
print(len(dataset))
print(dataset[0])

24853
(tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]]]), 3)


In [27]:
# 划分数据集
train_dataset, test_dataset = random_split(dataset, [0.75, 0.25])
print(len(train_dataset))
print(len(test_dataset))

18640
6213


In [29]:
# 创建数据加载器
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [30]:
for x, y in train_loader:
    print(x.shape, y.shape)
    print()
    break

torch.Size([32, 3, 64, 64]) torch.Size([32])



## 定义模型

In [31]:
from torch import nn, optim
model = nn.Sequential(
    # 第一层卷积-池化
    nn.Conv2d(3, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    # 第二层卷积-池化
    nn.Conv2d(8, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    # 全连接层
    nn.Linear(4096, 5)
)

In [32]:
print(model)

Sequential(
  (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=4096, out_features=5, bias=True)
)


In [33]:
data_iter = iter(train_loader)
input, target = next(data_iter)
print(input.shape, target.shape)

torch.Size([32, 3, 64, 64]) torch.Size([32])


In [34]:
# 前向传播测试
output = model(input)
print(output.shape)

torch.Size([32, 5])


## 模型训练

In [35]:
# 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Sequential(
  (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=4096, out_features=5, bias=True)
)

In [36]:
# 超参数设置
lr = 1e-3
epochs = 10

In [37]:
# 定义损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [41]:
# 模型训练
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for input, target in tqdm(train_loader):
        input, target = input.to(device), target.to(device)
        # 前向传播
        output = model(input)
        # 计算损失
        loss = loss_fn(output, target)
        # 反向传播
        loss.backward()
        # 更新参数
        optimizer.step()
        # 梯度清零
        optimizer.zero_grad()
        # 累加损失
        train_loss += loss.item()
    # 本epoch平均损失
    this_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch + 1} | Loss: {this_loss:.6f}")

100%|██████████| 582/582 [00:13<00:00, 43.47it/s]


Epoch 1 | Loss: 0.018699


100%|██████████| 582/582 [00:13<00:00, 44.00it/s]


Epoch 2 | Loss: 0.016249


100%|██████████| 582/582 [00:13<00:00, 43.97it/s]


...

Epoch 8 | Loss: 0.007138


100%|██████████| 582/582 [00:14<00:00, 40.65it/s]


Epoch 9 | Loss: 0.006685


100%|██████████| 582/582 [00:14<00:00, 39.25it/s]

Epoch 10 | Loss: 0.006121


## 模型测试

In [42]:
import matplotlib.pyplot as plt

In [46]:
# 1. 获取一批测试数据
test_iter = iter(test_loader)
images, labels = next(test_iter)
print(images.shape)
print(labels.shape)

torch.Size([32, 3, 64, 64])
torch.Size([32])


In [47]:
# 2. 前向传播
with torch.no_grad():
    images = images.to(device)
    outputs = model(images)
print(outputs.shape)

torch.Size([32, 5])


In [48]:
# 3. 将输出转换为分类预测标签
pred_labels = outputs.argmax(dim=1).cpu().numpy()
print(pred_labels.shape)

(32,)


In [49]:
# 4. 将输入图像数据，方便画图
images = images.permute(0, 2, 3, 1).cpu().numpy()
print(images.shape)

(32, 64, 64, 3)


In [ ]:
# 画图
fig, axes = plt.subplots(1, 10, figsize=(25, 4), sharex=True, sharey=True)
for i in range(10):
    axes[i].imshow(images[i])
    axes[i].axis("off")
    # 打印真实标签
    print(f"label-{i+1}: {labels[i]}")
    # 打印预测分类标签
    # 转换成中文分类
    pred_class = classification_names[pred_labels[i]]
    print(f"pred_label-{i+1}: {pred_labels[i]}, 分类名：{pred_class}")
    print()

plt.show()

In [51]:
# 遍历测试集，计算预测准确率
model.eval()
# 记录预测准确个数
test_correct_num = 0
with torch.no_grad():
    for input, target in test_loader:
        input = input.to(device)
        # 前向传播
        output = model(input)
        # 预测分类
        pred = output.argmax(dim=1).cpu()
        # 累加预测准确的个数
        test_correct_num += pred.eq(target).sum()
# 计算准确率
test_acc = test_correct_num / len(test_dataset)
print(test_acc)

tensor(0.9879)
